# Quick Evaluation: Truly Random Embedder (checkpoint_00)

This notebook performs a quick classification evaluation on the truly random embedder checkpoint to verify that random initialization performs poorly compared to trained embeddings.

**Dataset**: `artifacts/classification/mibig3/random_init/mibig3_bigcarp_embedder_00.pkl`  
**Evaluation**: Simple train/test split with BiLSTM classifier  
**Metrics**: Accuracy (A), Micro F1 (U), Macro F1 (R), Exact Match (O), AUC-ROC (C)

## Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, 
    roc_auc_score, hamming_loss, classification_report
)
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
warnings.filterwarnings('ignore')

# Add path for models
sys.path.append('/home/u5bb/han00.u5bb/workspace/cgrep')
from cgrep.models_multiclass import MultiLabelBiLSTMClassifier

print("✅ Imports completed")

✅ Imports completed


## Load Data

In [2]:
# Load the truly random embedder data (checkpoint_00)
data_path = "artifacts/classification/mibig3/random_init/mibig3_bigcarp_embedder_00.pkl"
print(f"Loading data from: {data_path}")

with open(data_path, 'rb') as f:
    df = pickle.load(f)

print(f"✅ Data loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Loading data from: artifacts/classification/mibig3/random_init/mibig3_bigcarp_embedder_00.pkl
✅ Data loaded: (2109, 6)
Columns: ['bgc_id', 'pfam_sequence', 'product_class', 'domain_sequence', 'tokenized_sequence', 'embeddings']

First few rows:


,bgc_id,pfam_sequence,product_class,domain_sequence,tokenized_sequence,embeddings
0,BGC0000001,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...",Polyketide,"[CMAS, PCMT, Fibrillarin, Methyltransf_23, Met...","[832, 6708, 3, 6171, 6175, 6177, 6173, 6158, 8...","[[-0.05702385, 1.6356643, 0.07847804, 1.010228..."
1,BGC0000002,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...",Polyketide,"[tRNA-synt_1c, UDPGT, Glyco_tran_28_C, Glyco_t...","[9380, 8840, 5122, 5135, 5145, 8619, 288, 285,...","[[-0.85113555, 0.8947296, -0.6335035, 1.173211..."
2,BGC0000003,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...",Polyketide,"[Carn_acyltransf, KR, ADH_zinc_N, Methyltransf...","[927, 5707, 158, 6171, 6163, 8892, 6177, 6158,...","[[0.0629913, -0.034768138, 0.5395474, -0.64251..."
3,BGC0000004,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, Thioesterase, PP-bind...","[6004, 8471, 8307, 8619, 6841, 6880, 352, 5694...","[[-0.43068644, -0.053428195, -1.3150823, -0.13..."
8,BGC0000010,"[PF07690, PF06609, PF00083, PF00106, PF08659, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, adh_short, KR, Epimer...","[6004, 8471, 8307, 9271, 5707, 4468, 6345, 927...","[[-0.43068644, -0.053428195, -1.3150823, -0.13..."


## Examine Embeddings

In [3]:
# Check embedding properties
sample_emb = df['embeddings'].iloc[0]
print(f"Sample embedding shape: {sample_emb.shape if hasattr(sample_emb, 'shape') else len(sample_emb) if isinstance(sample_emb, list) else 'Unknown'}")
print(f"Sample embedding type: {type(sample_emb)}")

if hasattr(sample_emb, 'shape'):
    emb_array = sample_emb
elif isinstance(sample_emb, list) and len(sample_emb) > 0:
    emb_array = np.array(sample_emb)
    print(f"Converted to numpy shape: {emb_array.shape}")

print(f"\nEmbedding statistics:")
print(f"  Mean: {emb_array.mean():.6f}")
print(f"  Std: {emb_array.std():.6f}")
print(f"  Min: {emb_array.min():.6f}")
print(f"  Max: {emb_array.max():.6f}")
print(f"\nFirst 10 values of first token:")
if emb_array.ndim == 2:
    print(emb_array[0, :10])
else:
    print(emb_array[:10])

Sample embedding shape: (99, 1280)
Sample embedding type: <class 'numpy.ndarray'>

Embedding statistics:
  Mean: 0.002541
  Std: 0.997337
  Min: -4.182577
  Max: 4.386367

First 10 values of first token:
[-0.05702385  1.6356643   0.07847804  1.0102282   0.09091879  0.15278354
 -0.509585   -1.6097577  -1.1512898   2.516006  ]


## Prepare Labels

In [4]:
def convert_product_classes_to_binary(df):
    """Convert semicolon-separated product_class to binary columns."""
    print("🔄 Converting product classes to binary columns...")
    
    df = df.copy()
    
    # Get all unique classes
    all_classes = set()
    for class_str in df['product_class'].dropna():
        if pd.isna(class_str) or class_str == '':
            continue
        classes = str(class_str).split(';')
        all_classes.update([cls.strip() for cls in classes if cls.strip()])
    
    all_classes = sorted(list(all_classes))
    print(f"   Found {len(all_classes)} unique classes: {all_classes}")
    
    # Create binary columns
    for class_name in all_classes:
        df[class_name] = df['product_class'].apply(
            lambda x: 1 if pd.notna(x) and class_name in str(x).split(';') else 0
        )
    
    # Show distribution
    print(f"\n   Class distribution:")
    for class_name in all_classes:
        count = df[class_name].sum()
        print(f"     {class_name}: {count:4d} samples ({count/len(df)*100:5.1f}%)")
    
    return df, all_classes

# Convert product classes to binary
df_prep, class_cols = convert_product_classes_to_binary(df)
print(f"\n✅ Binary conversion completed. Total classes: {len(class_cols)}")

🔄 Converting product classes to binary columns...
   Found 7 unique classes: ['Alkaloid', 'NRP', 'Other', 'Polyketide', 'RiPP', 'Saccharide', 'Terpene']

   Class distribution:
     Alkaloid:   68 samples (  3.2%)
     NRP:  727 samples ( 34.5%)
     Other:  291 samples ( 13.8%)
     Polyketide:  820 samples ( 38.9%)
     RiPP:  288 samples ( 13.7%)
     Saccharide:  148 samples (  7.0%)
     Terpene:  162 samples (  7.7%)

✅ Binary conversion completed. Total classes: 7


## Prepare Data for Training

In [5]:
def tensor_to_list(x):
    """Convert tensor to list if needed."""
    if hasattr(x, 'detach'):  # torch tensor
        return x.detach().cpu().numpy().tolist()
    elif hasattr(x, 'tolist'):  # numpy array
        return x.tolist()
    return x

# Prepare embeddings
print("🔄 Preparing embeddings...")
df_prep['embeddings'] = df_prep['embeddings'].apply(tensor_to_list)

# Create label strings for multi-label classification
label_strings = [";".join([c for c in class_cols if row[c]==1]) 
                 for _, row in df_prep.iterrows()]

print(f"✅ Data preparation completed")
print(f"   Total samples: {len(df_prep)}")
print(f"   Example label string: '{label_strings[0]}'")

# Check embedding dimensions
sample_emb = df_prep['embeddings'].iloc[0]
if isinstance(sample_emb, list) and len(sample_emb) > 0:
    if isinstance(sample_emb[0], list):
        emb_dim = len(sample_emb[0])
        seq_len = len(sample_emb)
    else:
        emb_dim = len(sample_emb)
        seq_len = 1
    
    print(f"   Embedding dimension: {emb_dim}")
    print(f"   Example sequence length: {seq_len}")

🔄 Preparing embeddings...
✅ Data preparation completed
   Total samples: 2109
   Example label string: 'Polyketide'
   Embedding dimension: 1280
   Example sequence length: 99


## Train/Test Split

In [6]:
# Simple train/test split (no CV for quick evaluation)
X = df_prep['embeddings'].tolist()
y = label_strings

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=None  # No stratify for simplicity
)

print(f"✅ Train/Test split completed")
print(f"   Training samples: {len(X_train)}")
print(f"   Testing samples: {len(X_test)}")

# Show some sample label distributions
from collections import Counter
train_labels = [label for labels in y_train for label in labels.split(';') if label]
test_labels = [label for labels in y_test for label in labels.split(';') if label]

print(f"\n   Training label distribution (top 5):")
for label, count in Counter(train_labels).most_common(5):
    print(f"     {label}: {count}")
    
print(f"\n   Test label distribution (top 5):")
for label, count in Counter(test_labels).most_common(5):
    print(f"     {label}: {count}")

✅ Train/Test split completed
   Training samples: 1687
   Testing samples: 422

   Training label distribution (top 5):
     Polyketide: 663
     NRP: 574
     Other: 238
     RiPP: 222
     Terpene: 134

   Test label distribution (top 5):
     Polyketide: 157
     NRP: 153
     RiPP: 66
     Other: 53
     Terpene: 28


## Train BiLSTM Model

In [7]:
import torch

# Check for GPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"\n🏋️  Training BiLSTM model on truly random embeddings...")
print(f"   Embedding dimension: {emb_dim}")
print(f"   Training samples: {len(X_train)}")

# Initialize model
model = MultiLabelBiLSTMClassifier(
    embed_dim=emb_dim, 
    hidden_dim=256,  # Smaller for quick evaluation
    num_layers=1,    # Smaller for quick evaluation  
    dropout_rate=0.2,
    pooling_strategy="mean",
    lr=1e-3,         # Higher LR for quick evaluation
    batch_size=32,
    early_stopping_patience=5,  # Shorter for quick evaluation
    max_epochs=20,   # Much shorter for quick evaluation
    random_seed=42,
    device=device
)

# Train model
print("\n🚀 Starting training...")
model.fit(X_train, y_train, n_folds=1, show_progress=True)

print("\n✅ Training completed!")

Using device: cuda:0

🏋️  Training BiLSTM model on truly random embeddings...
   Embedding dimension: 1280
   Training samples: 1687

🚀 Starting training...
Using device: cuda:0
Total samples: 1687
Number of classes: 7
Class names: ['Alkaloid' 'NRP' 'Other' 'Polyketide' 'RiPP' 'Saccharide' 'Terpene']
Using BiLSTM with 1 layers
Pooling strategy: mean
Train/Val samples: 1434
Test samples: 253
Using full training set without cross-validation (n_folds<=1)...


Training:  25%|██▌       | 5/20 [00:43<02:07,  8.52s/it]

  Epoch 5/20, Train Loss: 0.0386


Training:  50%|█████     | 10/20 [01:25<01:24,  8.48s/it]

  Epoch 10/20, Train Loss: 0.0050


Training:  75%|███████▌  | 15/20 [02:08<00:42,  8.55s/it]

  Epoch 15/20, Train Loss: 0.0022


Training: 100%|██████████| 20/20 [02:50<00:00,  8.53s/it]

  Epoch 20/20, Train Loss: 0.0010



Test Results (Best Model):
Test exact match accuracy: 0.6996
Test hamming loss: 0.0610
Test AUROC (micro-avg): 0.9504
Test AUROC (macro-avg): 0.9130
Test AUROC (weighted-avg): 0.9366

✅ Training completed!


## Evaluate Model

In [8]:
print("🔮 Making predictions on test set...")

# Get predictions
y_proba = model.predict_proba(X_test)
y_true = model.mlb.transform([s.split(';') if s else [] for s in y_test])
y_pred = (y_proba > 0.5).astype(int)

print(f"✅ Predictions completed")
print(f"   Prediction shape: {y_pred.shape}")
print(f"   Ground truth shape: {y_true.shape}")
print(f"   Number of classes: {len(model.mlb.classes_)}")
print(f"   Classes: {list(model.mlb.classes_)}")

🔮 Making predictions on test set...
✅ Predictions completed
   Prediction shape: (422, 7)
   Ground truth shape: (422, 7)
   Number of classes: 7
   Classes: ['Alkaloid', 'NRP', 'Other', 'Polyketide', 'RiPP', 'Saccharide', 'Terpene']


## Compute Metrics

In [9]:
def exact_match_accuracy(y_true, y_pred):
    """Exact match accuracy - all labels must be predicted correctly."""
    return np.mean(np.all(y_true == y_pred, axis=1))

def compute_comprehensive_metrics(y_true, y_pred, y_proba, class_names):
    """Compute comprehensive metrics."""
    metrics = {}
    
    # A: Exact match accuracy (O)
    metrics['exact_match_accuracy'] = exact_match_accuracy(y_true, y_pred)
    
    # U: Micro F1 
    metrics['micro_f1'] = f1_score(y_true, y_pred, average='micro')
    
    # R: Macro F1
    metrics['macro_f1'] = f1_score(y_true, y_pred, average='macro')
    
    # Weighted F1
    metrics['weighted_f1'] = f1_score(y_true, y_pred, average='weighted')
    
    # C: AUC-ROC metrics
    try:
        metrics['micro_auc'] = roc_auc_score(y_true.ravel(), y_proba.ravel())
    except ValueError:
        metrics['micro_auc'] = float('nan')
    
    try:
        metrics['macro_auc'] = roc_auc_score(y_true, y_proba, average='macro')
    except ValueError:
        metrics['macro_auc'] = float('nan')
    
    try:
        metrics['weighted_auc'] = roc_auc_score(y_true, y_proba, average='weighted')
    except ValueError:
        metrics['weighted_auc'] = float('nan')
    
    # Additional metrics
    metrics['hamming_loss'] = hamming_loss(y_true, y_pred)
    metrics['micro_precision'] = precision_score(y_true, y_pred, average='micro')
    metrics['micro_recall'] = recall_score(y_true, y_pred, average='micro')
    metrics['macro_precision'] = precision_score(y_true, y_pred, average='macro')
    metrics['macro_recall'] = recall_score(y_true, y_pred, average='macro')
    
    return metrics

# Compute all metrics
print("📊 Computing comprehensive metrics...")
metrics = compute_comprehensive_metrics(y_true, y_pred, y_proba, model.mlb.classes_)

print("\n✅ Metrics computed!")

📊 Computing comprehensive metrics...

✅ Metrics computed!


## Results Summary

In [10]:
print("\n" + "="*80)
print("🎯 TRULY RANDOM EMBEDDER (CHECKPOINT_00) EVALUATION RESULTS")
print("="*80)

print(f"\n📊 KEY METRICS (A-U-R-O-C format):")
print(f"   A (Exact Match Accuracy):     {metrics['exact_match_accuracy']:.4f}")
print(f"   U (Micro F1):                 {metrics['micro_f1']:.4f}")
print(f"   R (Macro F1):                 {metrics['macro_f1']:.4f}")
print(f"   O (Exact Match - same as A):  {metrics['exact_match_accuracy']:.4f}")
print(f"   C (Macro AUC-ROC):            {metrics['macro_auc']:.4f}")

print(f"\n📈 ADDITIONAL METRICS:")
print(f"   Weighted F1:                  {metrics['weighted_f1']:.4f}")
print(f"   Micro AUC-ROC:                {metrics['micro_auc']:.4f}")
print(f"   Weighted AUC-ROC:             {metrics['weighted_auc']:.4f}")
print(f"   Hamming Loss:                 {metrics['hamming_loss']:.4f}")

print(f"\n🔍 PRECISION & RECALL:")
print(f"   Micro Precision:              {metrics['micro_precision']:.4f}")
print(f"   Micro Recall:                 {metrics['micro_recall']:.4f}")
print(f"   Macro Precision:              {metrics['macro_precision']:.4f}")
print(f"   Macro Recall:                 {metrics['macro_recall']:.4f}")

print(f"\n🏷️  DATASET INFO:")
print(f"   Total test samples:           {len(y_test)}")
print(f"   Number of classes:            {len(model.mlb.classes_)}")
print(f"   Embedding dimension:          {emb_dim}")
print(f"   Model type:                   BiLSTM Classifier")

# Show prediction statistics
n_positive_true = y_true.sum()
n_positive_pred = y_pred.sum()
n_total_labels = y_true.size

print(f"\n📋 PREDICTION STATISTICS:")
print(f"   True positive labels:         {n_positive_true} / {n_total_labels} ({n_positive_true/n_total_labels*100:.1f}%)")
print(f"   Predicted positive labels:    {n_positive_pred} / {n_total_labels} ({n_positive_pred/n_total_labels*100:.1f}%)")
print(f"   Correct predictions:          {np.sum(y_true == y_pred)} / {n_total_labels} ({np.sum(y_true == y_pred)/n_total_labels*100:.1f}%)")


🎯 TRULY RANDOM EMBEDDER (CHECKPOINT_00) EVALUATION RESULTS

📊 KEY METRICS (A-U-R-O-C format):
   A (Exact Match Accuracy):     0.7488
   U (Micro F1):                 0.8347
   R (Macro F1):                 0.6943
   O (Exact Match - same as A):  0.7488
   C (Macro AUC-ROC):            0.9397

📈 ADDITIONAL METRICS:
   Weighted F1:                  0.8321
   Micro AUC-ROC:                0.9658
   Weighted AUC-ROC:             0.9593
   Hamming Loss:                 0.0535

🔍 PRECISION & RECALL:
   Micro Precision:              0.8636
   Micro Recall:                 0.8077
   Macro Precision:              0.7430
   Macro Recall:                 0.6647

🏷️  DATASET INFO:
   Total test samples:           422
   Number of classes:            7
   Embedding dimension:          1280
   Model type:                   BiLSTM Classifier

📋 PREDICTION STATISTICS:
   True positive labels:         494 / 2954 (16.7%)
   Predicted positive labels:    462 / 2954 (15.6%)
   Correct predictions:      

## Per-Class Performance

In [11]:
print("\n" + "-"*60)
print("📊 PER-CLASS PERFORMANCE BREAKDOWN")
print("-"*60)

class_names = model.mlb.classes_
per_class_results = []

for i, class_name in enumerate(class_names):
    if i < y_true.shape[1]:
        class_true = y_true[:, i]
        class_pred = y_pred[:, i]
        class_proba = y_proba[:, i]
        
        support = int(np.sum(class_true))
        
        # Calculate metrics if both classes present
        if len(np.unique(class_true)) > 1:
            try:
                auc = roc_auc_score(class_true, class_proba)
                f1 = f1_score(class_true, class_pred)
                precision = precision_score(class_true, class_pred)
                recall = recall_score(class_true, class_pred)
            except:
                auc = f1 = precision = recall = float('nan')
        else:
            auc = f1 = precision = recall = float('nan')
        
        per_class_results.append({
            'class': class_name,
            'support': support,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'auc': auc,
            'frequency': support / len(y_test)
        })

# Sort by support (most common first)
per_class_results.sort(key=lambda x: x['support'], reverse=True)

print(f"{'Class':<12} {'Support':<8} {'Freq%':<7} {'F1':<7} {'Prec':<7} {'Recall':<7} {'AUC':<7}")
print("-" * 60)

for result in per_class_results:
    print(f"{result['class']:<12} "
          f"{result['support']:<8} "
          f"{result['frequency']*100:<6.1f}% "
          f"{result['f1']:<7.3f} "
          f"{result['precision']:<7.3f} "
          f"{result['recall']:<7.3f} "
          f"{result['auc']:<7.3f}")

# Calculate average performance for non-NaN values
valid_results = [r for r in per_class_results if not np.isnan(r['f1'])]
if valid_results:
    avg_f1 = np.mean([r['f1'] for r in valid_results])
    avg_auc = np.mean([r['auc'] for r in valid_results if not np.isnan(r['auc'])])
    print("-" * 60)
    print(f"{'Average':<12} {'N/A':<8} {'N/A':<7} {avg_f1:<7.3f} {'N/A':<7} {'N/A':<7} {avg_auc:<7.3f}")



------------------------------------------------------------
📊 PER-CLASS PERFORMANCE BREAKDOWN
------------------------------------------------------------
Class        Support  Freq%   F1      Prec    Recall  AUC    
------------------------------------------------------------
Polyketide   157      37.2  % 0.919   0.940   0.898   0.981  
NRP          153      36.3  % 0.880   0.898   0.863   0.964  
RiPP         66       15.6  % 0.893   0.982   0.818   0.981  
Other        53       12.6  % 0.630   0.618   0.642   0.881  
Saccharide   28       6.6   % 0.618   0.630   0.607   0.930  
Terpene      28       6.6   % 0.755   0.800   0.714   0.972  
Alkaloid     9        2.1   % 0.167   0.333   0.111   0.870  
------------------------------------------------------------
Average      N/A      N/A     0.694   N/A     N/A     0.940  


## Summary and Conclusions

In [12]:
print("\n" + "="*80)
print("🔬 ANALYSIS: TRULY RANDOM EMBEDDER PERFORMANCE")
print("="*80)

print(f"\n💡 KEY FINDINGS:")

# Assess performance levels
performance_assessment = []

if metrics['exact_match_accuracy'] < 0.05:
    performance_assessment.append("❌ VERY POOR exact match accuracy (<5%)")
elif metrics['exact_match_accuracy'] < 0.15:
    performance_assessment.append("⚠️  POOR exact match accuracy (<15%)")
else:
    performance_assessment.append("✅ Unexpectedly good exact match accuracy")

if metrics['macro_f1'] < 0.1:
    performance_assessment.append("❌ VERY POOR macro F1 (<10%)")
elif metrics['macro_f1'] < 0.3:
    performance_assessment.append("⚠️  POOR macro F1 (<30%)")
else:
    performance_assessment.append("✅ Unexpectedly good macro F1")

if not np.isnan(metrics['macro_auc']):
    if metrics['macro_auc'] < 0.6:
        performance_assessment.append("❌ VERY POOR macro AUC (<60%)")
    elif metrics['macro_auc'] < 0.75:
        performance_assessment.append("⚠️  POOR macro AUC (<75%)")
    else:
        performance_assessment.append("✅ Unexpectedly good macro AUC")

for assessment in performance_assessment:
    print(f"   {assessment}")

print(f"\n📊 QUICK COMPARISON FORMAT (A-U-R-O-C):")
print(f"   Random Embedder: {metrics['exact_match_accuracy']:.3f}-{metrics['micro_f1']:.3f}-{metrics['macro_f1']:.3f}-{metrics['exact_match_accuracy']:.3f}-{metrics['macro_auc']:.3f}")

print(f"\n🎯 EXPECTED BEHAVIOR:")
print(f"   ✓ Truly random embeddings should perform poorly")
print(f"   ✓ This confirms the earlier analysis that trained embeddings")
print(f"     (even from 'random_init') were actually learned representations")
print(f"   ✓ Performance difference should be dramatic compared to trained models")

print(f"\n💾 COMPARISON DATA:")
print(f"   Use this as baseline to compare against:")
print(f"   - mibig3_bigcarp_embedder.pkl (trained 'random' init)")
print(f"   - mibig3_bigcarp_last.pkl (trained last layer)")
print(f"   - ESM embeddings")

print(f"\n" + "="*80)


🔬 ANALYSIS: TRULY RANDOM EMBEDDER PERFORMANCE

💡 KEY FINDINGS:
   ✅ Unexpectedly good exact match accuracy
   ✅ Unexpectedly good macro F1
   ✅ Unexpectedly good macro AUC

📊 QUICK COMPARISON FORMAT (A-U-R-O-C):
   Random Embedder: 0.749-0.835-0.694-0.749-0.940

🎯 EXPECTED BEHAVIOR:
   ✓ Truly random embeddings should perform poorly
   ✓ This confirms the earlier analysis that trained embeddings
     (even from 'random_init') were actually learned representations
   ✓ Performance difference should be dramatic compared to trained models

💾 COMPARISON DATA:
   Use this as baseline to compare against:
   - mibig3_bigcarp_embedder.pkl (trained 'random' init)
   - mibig3_bigcarp_last.pkl (trained last layer)
   - ESM embeddings

